# 🎓 GENI YOLO Training Notebook

Train YOLO11 on your 12 GENI training images

**Steps:**
1. Upload your training data
2. Augment images (12 → 120+)
3. Convert to YOLO format
4. Train YOLO11s model
5. Download trained model

**Runtime:** Make sure GPU is enabled (Runtime → Change runtime type → T4 GPU)

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install ultralytics opencv-python pillow

## 📥 Step 1: Upload Training Data

Upload your `training-data` folder:
- `labels.json`
- `images/` folder with 12 images

In [ ]:
import os
import json
import shutil
from pathlib import Path

# Create directories
!mkdir -p data/original
!mkdir -p data/augmented
!mkdir -p data/yolo/images/train
!mkdir -p data/yolo/images/val
!mkdir -p data/yolo/labels/train
!mkdir -p data/yolo/labels/val

print("✅ Directories created")
print("📤 Upload your training data now:")
print("   1. Click folder icon on left")
print("   2. Upload labels.json")
print("   3. Upload all images from images/ folder")

## 🎨 Step 2: Image Augmentation

Multiply your 12 images with transformations

In [ ]:
from PIL import Image, ImageEnhance, ImageFilter
import random

def augment_image(img_path, output_dir, base_name, labels):
    """Apply augmentations to a single image"""
    img = Image.open(img_path)
    augmented = []
    
    # Original
    output_path = os.path.join(output_dir, f"{base_name}_original.jpg")
    img.save(output_path)
    augmented.append(output_path)
    
    # Rotate variations
    for angle in [5, -5, 10, -10]:
        rotated = img.rotate(angle, expand=False, fillcolor='white')
        output_path = os.path.join(output_dir, f"{base_name}_rot{angle}.jpg")
        rotated.save(output_path)
        augmented.append(output_path)
    
    # Flip horizontal
    flipped = img.transpose(Image.FLIP_LEFT_RIGHT)
    output_path = os.path.join(output_dir, f"{base_name}_flip_h.jpg")
    flipped.save(output_path)
    augmented.append(output_path)
    
    # Flip vertical
    flipped = img.transpose(Image.FLIP_TOP_BOTTOM)
    output_path = os.path.join(output_dir, f"{base_name}_flip_v.jpg")
    flipped.save(output_path)
    augmented.append(output_path)
    
    # Brightness variations
    for factor in [0.8, 1.2]:
        enhancer = ImageEnhance.Brightness(img)
        bright = enhancer.enhance(factor)
        output_path = os.path.join(output_dir, f"{base_name}_bright{int(factor*100)}.jpg")
        bright.save(output_path)
        augmented.append(output_path)
    
    # Blur
    blurred = img.filter(ImageFilter.GaussianBlur(radius=1))
    output_path = os.path.join(output_dir, f"{base_name}_blur.jpg")
    blurred.save(output_path)
    augmented.append(output_path)
    
    return augmented

# Load labels
with open('labels.json', 'r') as f:
    labels = json.load(f)

print(f"📊 Found {len(labels)} original images")
print("🎨 Starting augmentation...\n")

augmented_labels = []
total_augmented = 0

for label in labels:
    img_path = label['filename']
    if not os.path.exists(img_path):
        print(f"⚠️  Skipping {img_path} (not found)")
        continue
    
    base_name = Path(img_path).stem
    
    augmented_paths = augment_image(
        img_path,
        'data/augmented',
        base_name,
        label
    )
    
    # Create labels for each augmented image
    for aug_path in augmented_paths:
        augmented_labels.append({
            'filename': aug_path,
            'category': label['category'],
            'description': label.get('description', ''),
        })
        total_augmented += 1
    
    print(f"✅ {base_name}: {len(augmented_paths)} variations")

print(f"\n✅ Augmentation complete!")
print(f"   Original: {len(labels)}")
print(f"   Augmented: {total_augmented}")
print(f"   Multiplier: {total_augmented/len(labels):.1f}x")

## 🔄 Step 3: Convert to YOLO Format

YOLO needs:
- Images in `images/train` and `images/val`
- Labels in `labels/train` and `labels/val`
- `data.yaml` config file

In [ ]:
import random

# Get unique categories
categories = list(set(label['category'] for label in augmented_labels))
categories.sort()

print(f"📋 Categories: {categories}")

# Split train/val (90/10)
random.shuffle(augmented_labels)
split_idx = int(len(augmented_labels) * 0.9)
train_labels = augmented_labels[:split_idx]
val_labels = augmented_labels[split_idx:]

print(f"\n📊 Dataset split:")
print(f"   Train: {len(train_labels)} images")
print(f"   Val: {len(val_labels)} images")

# For image classification (not object detection), 
# we just copy images to category folders
# But GENI needs object detection, so we'll create dummy bounding boxes

def create_yolo_label(category, categories, img_width=640, img_height=640):
    """Create YOLO label (whole image bounding box)"""
    class_id = categories.index(category)
    # YOLO format: class_id x_center y_center width height (normalized 0-1)
    # Whole image: center at 0.5, 0.5, width and height 1.0
    return f"{class_id} 0.5 0.5 1.0 1.0\n"

# Copy images and create labels
for split_name, split_labels in [('train', train_labels), ('val', val_labels)]:
    for label in split_labels:
        # Copy image
        src_img = label['filename']
        dst_img = os.path.join('data/yolo/images', split_name, os.path.basename(src_img))
        shutil.copy(src_img, dst_img)
        
        # Create YOLO label file
        label_filename = Path(os.path.basename(src_img)).stem + '.txt'
        label_path = os.path.join('data/yolo/labels', split_name, label_filename)
        
        with open(label_path, 'w') as f:
            f.write(create_yolo_label(label['category'], categories))

print("\n✅ YOLO dataset created!")

In [ ]:
# Create data.yaml
data_yaml = f"""train: data/yolo/images/train
val: data/yolo/images/val

nc: {len(categories)}
names: {categories}
"""

with open('data/data.yaml', 'w') as f:
    f.write(data_yaml)

print("✅ data.yaml created:")
print(data_yaml)

## 🚀 Step 4: Train YOLO Model

Train YOLO11s on your dataset

In [ ]:
from ultralytics import YOLO

# Load YOLO11s model
model = YOLO('yolo11s.pt')

# Train
results = model.train(
    data='data/data.yaml',
    epochs=60,          # Good for small datasets
    imgsz=640,          # Image size
    batch=16,           # Batch size
    device=0,           # GPU
    patience=10,        # Early stopping
    save=True,
    project='geni_model',
    name='pill_detection',
)

print("\n✅ Training complete!")
print("📂 Model saved to: runs/detect/train/weights/best.pt")

## 📊 Step 5: View Results

In [ ]:
from IPython.display import Image, display

# Display training results
display(Image('geni_model/pill_detection/results.png'))
display(Image('geni_model/pill_detection/confusion_matrix.png'))

## 🧪 Step 6: Test Model

In [ ]:
# Load best model
model = YOLO('geni_model/pill_detection/weights/best.pt')

# Test on validation images
results = model.predict(
    source='data/yolo/images/val',
    save=True,
    project='geni_model',
    name='predictions',
)

print("✅ Predictions saved to: geni_model/predictions/")

## 📥 Step 7: Download Trained Model

In [ ]:
# Zip the model
!zip -r geni_pill_model.zip geni_model/pill_detection/weights/best.pt

print("✅ Model zipped!")
print("📥 Download: geni_pill_model.zip from the files panel")
print("")
print("🎯 Next steps:")
print("   1. Download geni_pill_model.zip")
print("   2. Extract best.pt")
print("   3. Move to GENI backend")
print("   4. Update yolo.ts to use this model")
print("   5. Test in GENI!")